# TEM Tango Control — Client Tutorial

This notebook demonstrates how to connect to and interact with the TEM Tango devices.

All device servers must be running **before** executing this notebook.

Start the HAADF detector:

```bash
python -m tango.test_context src.detectors.HAADF.HAADF --host 127.0.0.1 --port 8888
```

Start the Microscope (in a separate terminal):

```bash
uv run python -m tango.test_context src.Microscope.Microscope \
  --host 127.0.0.1 \
  --port 8889 \
  --prop "{
    'haadf_device_address': 'tango://127.0.0.1:8888/test/nodb/haadf#dbase=no',
    'autoscript_host_ip': '10.46.217.242',
    'autoscript_host_port': 9090
  }"
```

---

If You Get “Address Already in Use”

It means something is already running on that port (likely from a previous session).

To check what is using a port:

```bash
lsof -i :8888
```

Then kill it using the PID shown:

```bash
kill -9 <PID>
```

Quick one-liner:

```bash
kill -9 $(lsof -t -i:8888)
```

Repeat for port `8889` if needed.

---

If a notebook crashes, `tango.test_context` may continue running in the background.
Always stop the previous process before restarting the device server.




## 1. Connect to devices

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tango

# --nodb mode — use full tango:// URL with port and #dbase=no suffix
haadf_proxy = tango.DeviceProxy("tango://127.0.0.1:8888/test/nodb/haadf#dbase=no")
microscope_proxy = tango.DeviceProxy("tango://127.0.0.1:8889/test/nodb/microscope#dbase=no")



In [ ]:
print('HAADF state     :', haadf_proxy.state())
print('Microscope state:', microscope_proxy.state())

## 2. Inspect device attributes and commands

In [ ]:
print('--- HAADF attributes ---')
for attr in haadf_proxy.get_attribute_list():
    print(f'  {attr}')

print('\n--- Microscope commands ---')
for cmd in microscope_proxy.get_command_list():
    print(f'  {cmd}')

## 3. Configure HAADF detector settings

In [ ]:
haadf_proxy.dwell_time 

In [ ]:
haadf_proxy.dwell_time   = 1e-6   # 1 µs
haadf_proxy.image_width  = 1024
haadf_proxy.image_height = 1024

print('dwell_time  :', haadf_proxy.dwell_time)
print('image_width :', haadf_proxy.image_width)
print('image_height:', haadf_proxy.image_height)

## 4. Acquire a HAADF image

In [ ]:
# get_image returns DevEncoded = (json_metadata_str, raw_bytes)
json_meta, raw_bytes = microscope_proxy.get_image('haadf')

metadata  = dict(json.loads(json_meta))
image = np.frombuffer(raw_bytes, dtype=metadata['dtype']).reshape(metadata['shape'])

print('Metadata:', metadata)
print('Image shape:', image.shape)
print('Image dtype:', image.dtype)

## 5. Display the image

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray', interpolation='none')
ax.set_title(f"HAADF — dwell {metadata['dwell_time']*1e6:.1f} µs")
ax.axis('off')
plt.tight_layout()
plt.show()

### Sidpy dataset

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

import sidpy
print('sidpy version: ', sidpy.__version__)

In [ ]:
dataset = sidpy.Dataset.from_array(image , name='random')

# set dimesnions
dataset.set_dimension(0, sidpy.Dimension(np.arange(image.shape[0])*.02, 'x'))
dataset.set_dimension(1, sidpy.Dimension(np.arange(image.shape[0])*.02, 'y'))


# set the dataset level plotting metadata
dataset.data_type = 'image'
dataset.units = 'counts'
dataset.quantity = 'intensity'
dataset.title = 'random'

# handle one dimension of the data
dataset.set_dimension(0, sidpy.Dimension(np.arange(dataset.shape[0])*.02, 'x'))
dataset.x.dimension_type = 'spatial'
dataset.x.units = 'nm'
dataset.x.quantity = 'distance'

# handle another dimension of the data

dataset.set_dimension(1, sidpy.Dimension(np.arange(dataset.shape[1])*.02, 'y'))
dataset.y.dimension_type = 'spatial'
dataset.y.units = 'nm'
dataset.y.quantity = 'distance'
view = dataset.plot(scale_bar=True)

In [ ]:
dataset.metadata = metadata

In [ ]:
view = dataset.plot(scale_bar=True)

## 6. Advanced acquisition (multi-detector)

> **Not yet implemented.** See `src/acquisition/advanced_acquisition.py` for design notes.